# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. когда мы делим массив нампай, то мы получим в первом(а/2) - [1., 2., 3.], а во втором случае получим - [0, 1, 1], то есть // оставляет только дробную часть
2. у нас есть m = [1, 1, 1, 1][1, 1, 1, 1][1, 1, 1, 1]; а в принте выйдет - [4, 4, 4] - потому что axis=1 считает и проходится по каждому столбцу
3. при сложений двух нампай массивов то мы получим - [0, 2, 4] а при reshape мы получим 3 строки и 1 столбец, поучится = [0][2][4]
4. при np.sqrt(x)- мы получим [1, 2, 3], а сумма будет = 6
5. сам массив - [1, 2, 3, 4], **2 = [1, 4, 9, 16], .mean = 7.5, полчется как ответ выйдет - 7.5

In [1]:
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())

[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [3]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [4]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [5]:
# 1
sep = amount[clean & (month == "2026-09")].sum()
oct_ = amount[clean & (month == "2026-10")].sum()
print(f"выручка за сентябрь: {round(sep)}, окятбрь: {round(oct_)}, изменение: {(oct_ - sep)/sep*100}%")

выручка за сентябрь: 35243410, окятбрь: 34419520, изменение: -2.337713632137185%


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [6]:
# 2
cats = np.unique(cat)
months = np.array(["2026-09", "2026-10"])

table_cat = []

for c in cats:
  sep = amount[clean & (month == "2026-09") & (cat == c)].sum()
  oct_ = amount[clean & (month == "2026-10") & (cat == c)].sum()
  table_cat.append([sep, oct_])

table_cat = np.array(table_cat)
print(table_cat.shape)

print(f"{'Категория':<12} {'Сентябрь':>12} {'Октябрь':>12}")
for c, row in zip(cats, table_cat):
    print(f"{c:<12} {row[0]:>12,.0f} {row[1]:>12,.0f}")

total = table_cat.sum(axis=1)
best = total.argmax()
print(f"\nБольше всего выручки: {cats[best]} — {total[best]} tg")

sep_col = table_cat[:, 0]
oct_col = table_cat[:, 1]
change = (oct_col - sep_col) / sep_col * 100
worst = change.argmin()
print(f"Сильнее всех упала: {cats[worst]} — {abs(change[worst])}%")

(5, 2)
Категория        Сентябрь      Октябрь
Дом             4,997,730    5,125,790
Книги           1,065,300    1,140,910
Красота         2,610,950    2,678,990
Одежда          9,167,100    8,505,730
Электроника    17,402,330   16,968,100

Больше всего выручки: Электроника — 34370430.0 tg
Сильнее всех упала: Одежда — 7.214604400519248%


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [7]:
# 3
table_c = []
for c in cats:
  sep1 = sum(clean & (cat == c) & (month == "2026-09"))
  oct1 = sum(clean & (cat == c) & (month == "2026-10"))
  table_c.append([sep1, oct1])
table_c = np.array(table_c)

avg_check = table_cat / table_c

print(f"{'Категория':<12} {'Заказов сен':>12} {'Заказов окт':>12} {'Чек сен':>10} {'Чек окт':>10}")
for c, n, a in zip(cats, table_c, avg_check):
    print(f"{c:<12} {n[0]:>12} {n[1]:>12} {a[0]:>10,.0f} {a[1]:>10,.0f}")

oct_avg = avg_check[:, 1]
best = oct_avg.argmax()
print(f"\nСамый высокий средний чек в октябре: {cats[best]} — {oct_avg[best]:,.0f} тг")


Категория     Заказов сен  Заказов окт    Чек сен    Чек окт
Дом                   476          476     10,499     10,768
Книги                 265          299      4,020      3,816
Красота               422          435      6,187      6,159
Одежда                678          652     13,521     13,046
Электроника           531          505     32,773     33,600

Самый высокий средний чек в октябре: Электроника — 33,600 тг


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [8]:
# 4
cities = np.unique(city)
months = np.array(["2026-09", "2026-10"])

table = np.array([[amount[clean & (city == c) & (month == m)].sum() for m in months]
                  for c in cities])


share = table / table.sum(axis=0)

print(f"{'Город':<12} {'Сентябрь':>10} {'Октябрь':>10}")
for c, s in zip(cities, share):
    print(f"{c:<12} {s[0] * 100:>9.1f}% {s[1] * 100:>9.1f}%")


print("\nСумма по столбцам:", share.sum(axis=0))

# Шымкент
shym = share[cities == "Шымкент"][0]
print(f"Шымкент: {shym[0] * 100:.1f}% -> {shym[1] * 100:.1f}% "
      f"(изменение {(shym[1] - shym[0]) * 100:+.1f} %)")

Город          Сентябрь    Октябрь
Актобе             7.3%       7.4%
Алматы            41.1%      42.5%
Астана            25.6%      25.7%
Атырау             2.6%       2.8%
Караганда         10.4%       9.8%
Шымкент           12.9%      11.7%

Сумма по столбцам: [1. 1.]
Шымкент: 12.9% -> 11.7% (изменение -1.2 п.п.)


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [13]:
# 5
diff = table[:, 1] - table[:, 0]
total_diff = diff.sum()
contrib = diff / total_diff

print(f"Общее изменение выручки: {total_diff:,.0f} тг\n")
print(f"{'Город':<12} {'Изменение':>12} {'Вклад':>8}")
for c, d, k in zip(cities, diff, contrib):
    print(f"{c:<12} {d:>12,.0f} {k * 100:>7.1f}%")

print("\nПроверка, сумма вкладов:", contrib.sum())

shym = contrib[cities == "Шымкент"][0]
print(f"Шымкент даёт {shym * 100:.1f}% всего падения")

Общее изменение выручки: -823,890 тг

Город           Изменение    Вклад
Актобе            -36,540     4.4%
Алматы            141,950   -17.2%
Астана           -183,690    22.3%
Атырау             45,870    -5.6%
Караганда        -276,840    33.6%
Шымкент          -514,640    62.5%

Проверка, сумма вкладов: 1.0
Шымкент даёт 62.5% всего падения


В Шымкенте самый высокий процент отмен и самая медленная доставка, поэтому начинать стоит с проверки логистики.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

код падает, потому что у нас table = (6, 2), то есть 6 городов и 2 месяца, но при делении (6, 2)/(6, ) у нас будет ошибка, (6,) - это значит shape of table.sum(axis=1). У нас axis=1 считает по столбцам каждый город и в итоге мы получаем таблицу с shape (6,). и для решения этой задачи мы можем использовать keepdims=True

In [14]:
# 6
share = table / table.sum(axis=1, keepdims=True)

print(f"{'Город':<12} {'Сентябрь':>10} {'Октябрь':>10}")
for c, s in zip(cities, share):
    print(f"{c:<12} {s[0] * 100:>9.1f}% {s[1] * 100:>9.1f}%")

print("\nПроверка, сумма по строкам:", share.sum(axis=1))

Город          Сентябрь    Октябрь
Актобе            50.4%      49.6%
Алматы            49.8%      50.2%
Астана            50.5%      49.5%
Атырау            48.8%      51.2%
Караганда         52.0%      48.0%
Шымкент           53.0%      47.0%

Проверка, сумма по строкам: [1. 1. 1. 1. 1. 1.]


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [19]:
# 7
x = amount[clean]
z = (x - x.mean()) / x.std()

outliers = np.abs(z) > 3
print(f"Заказы с |z|>3 : {outliers.sum()}")
print(f"Доля: {outliers.mean() * 100:.1f}%")
print(f"\nСреднее z: {round(z.mean(), 2)}")
print(f"Std z: {round(z.std(), 4)}")

Заказы с |z|>3 : 87
Доля: 1.8%

Среднее z: 0.0
Std z: 1.0


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [22]:
# 8
mask = clean & ~np.isnan(days)

X = np.column_stack([amount[mask], days[mask]])
print("Форма X:", X.shape)
print("До: min =", X.min(axis=0), " max =", X.max(axis=0))

# Нормализация(min-max)
X_norm = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))

print("После: min =", X_norm.min(axis=0), " max =", X_norm.max(axis=0))

Форма X: (4595, 2)
До: min = [720.   1.]  max = [1.3904e+05 8.0000e+00]
После: min = [0. 0.]  max = [1. 1.]


нормализация нужна чтобы все данные весили одинаково, то есть, чтобы  модель не решила, что какой-то из признаков важнее другого из-за того что какой-то признак имеет более высокие значения(цифры). Например в нашем датасете моделька могла подумать, что признак amount важнее чем delivery-days, хотя они одинаково нужны и важны.  